<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# CELL 1: DATASET LOADING & GOOGLE DRIVE CACHING
# =====================================================================
import os
import torch
from torchvision.datasets import CIFAR10
from torchvision.transforms import Resize, ToTensor, Normalize, Compose
from torch.utils.data import DataLoader, Subset

# 1. Detect Google Colab and Mount Google Drive
try:
    from google.colab import drive
    print("Google Colab detected. Mounting Google Drive...")
    drive.mount('/content/drive')
    dataset_root = '/content/drive/MyDrive/Colab_Datasets/CIFAR10'
    os.makedirs(dataset_root, exist_ok=True)
    print(f"Dataset directory: {dataset_root}")
except ImportError:
    print("Local environment detected. Storing dataset locally.")
    dataset_root = './data'

# 2. Define Filtered Dataset Loader Function
def get_filtered_loaders(classes_to_keep=[0, 1, 2], batch_size=32, train_samples_per_class=1000, test_samples_per_class=200):
    transform = Compose([
        Resize(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Download dataset to Google Drive (only runs the first time)
    train_dataset = CIFAR10(root=dataset_root, train=True, download=True, transform=transform)
    test_dataset = CIFAR10(root=dataset_root, train=False, download=True, transform=transform)

    def filter_indices(dataset, samples_per_class):
        indices = []
        counts = {c: 0 for c in classes_to_keep}
        for idx, (_, label) in enumerate(dataset):
            if label in counts and counts[label] < samples_per_class:
                indices.append(idx)
                counts[label] += 1
            if all(count >= samples_per_class for count in counts.values()):
                break
        return indices

    train_idx = filter_indices(train_dataset, train_samples_per_class)
    test_idx = filter_indices(test_dataset, test_samples_per_class)

    label_mapping = {orig: new for new, orig in enumerate(classes_to_keep)}

    class MappedSubset(torch.utils.data.Dataset):
        def __init__(self, subset, mapping):
            self.subset = subset
            self.mapping = mapping

        def __getitem__(self, index):
            img, target = self.subset[index]
            return img, self.mapping[target]

        def __len__(self):
            return len(self.subset)

    train_sub = MappedSubset(Subset(train_dataset, train_idx), label_mapping)
    test_sub = MappedSubset(Subset(test_dataset, test_idx), label_mapping)

    train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_sub, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

# 3. Load and Cache the Data
classes_to_keep = [0, 1, 2]  # Airplane, Automobile, Bird
class_names = ["Airplane", "Automobile", "Bird"]

print(f"\nLoading and filtering dataset for classes: {class_names}...")
train_loader, test_loader = get_filtered_loaders(
    classes_to_keep=classes_to_keep,
    train_samples_per_class=1000,
    test_samples_per_class=200,
    batch_size=32
)
print(f"Dataset ready. Train size: {len(train_loader.dataset)}, Test size: {len(test_loader.dataset)}")

Google Colab detected. Mounting Google Drive...
Mounted at /content/drive
Dataset directory: /content/drive/MyDrive/Colab_Datasets/CIFAR10

Loading and filtering dataset for classes: ['Airplane', 'Automobile', 'Bird']...
Dataset ready. Train size: 3000, Test size: 600


In [2]:
# =====================================================================
# CELL 2: CORE DISTRIBUTED GRAPH ARCHITECTURE
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque

# 1. Define Compression Interface & Coders
class CompressionInterface(nn.Module):
    def __init__(self):
        super().__init__()
    def compress(self, x: torch.Tensor, target_ratio: int) -> torch.Tensor:
        raise NotImplementedError
    def decompress(self, x: torch.Tensor, target_ratio: int, target_shape: tuple) -> torch.Tensor:
        raise NotImplementedError

class SpatialDownsamplingCoder(CompressionInterface):
    """Compresses spatially. Transmits 32-bit floats."""
    def compress(self, x: torch.Tensor, target_ratio: int) -> torch.Tensor:
        if target_ratio == 1:
            return x
        scale_factor = 1.0 / (target_ratio ** 0.5)
        return F.interpolate(x, scale_factor=scale_factor, mode='bilinear', align_corners=False)

    def decompress(self, x: torch.Tensor, target_ratio: int, target_shape: tuple) -> torch.Tensor:
        if target_ratio == 1:
            return x
        return F.interpolate(x, size=(target_shape[2], target_shape[3]), mode='bilinear', align_corners=False)


class STEQuantizer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, num_bits):
        x_bounded = torch.sigmoid(x)
        levels = (2 ** num_bits) - 1
        quantized = torch.round(x_bounded * levels) / levels
        return quantized
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


class TrainableVDDIBCoder(CompressionInterface):
    """Compresses channels dynamically, normalizes, and quantizes to 8-bit integers."""
    def __init__(self, num_bits: int = 8):
        super().__init__()
        self.num_bits = num_bits
        self.encoder = None
        self.bn = None  # Group Normalization to prevent train/eval mismatch
        self.decoder = None
        self.is_initialized = False

    def _lazy_init(self, in_channels, compressed_channels, device):
        self.encoder = nn.Conv2d(in_channels, compressed_channels, kernel_size=1).to(device)
        # GroupNorm with 1 group acts as LayerNorm, requiring no running statistics
        self.bn = nn.GroupNorm(1, compressed_channels).to(device)
        self.decoder = nn.Conv2d(compressed_channels, in_channels, kernel_size=1).to(device)
        self.is_initialized = True

    def compress(self, x: torch.Tensor, target_ratio: int) -> torch.Tensor:
        in_channels = x.shape[1]
        if target_ratio == 1:
            return x

        reduction_factor = max(1, target_ratio // 4)
        compressed_channels = max(1, in_channels // reduction_factor)

        if not self.is_initialized or self.encoder.weight.shape[1] != in_channels:
            self._lazy_init(in_channels, compressed_channels, x.device)

        encoded = self.encoder(x)
        normalized = self.bn(encoded)  # Stable normalization with no train/eval discrepancy
        quantized = STEQuantizer.apply(normalized, self.num_bits)
        return quantized

    def decompress(self, x: torch.Tensor, target_ratio: int, target_shape: tuple) -> torch.Tensor:
        if target_ratio == 1:
            return x
        return self.decoder(x)


# 2. Define Communication Edge & Device Nodes
class CommunicationEdge(nn.Module):
    def __init__(self, source_id: str, target_id: str, bandwidth_bps: float = 10e6):
        super().__init__()
        self.source_id = source_id
        self.target_id = target_id
        self.bandwidth_bps = bandwidth_bps
        self.coder = SpatialDownsamplingCoder()
        self.target_ratio = 1
        self.accumulated_tx_time = 0.0

    def reset_tracker(self):
        self.accumulated_tx_time = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        original_shape = x.shape
        is_no_compression = (self.target_ratio == 1)

        if is_no_compression:
            total_bits = x.numel() * 32
            self.accumulated_tx_time += total_bits / self.bandwidth_bps
            return x

        compressed_x = self.coder.compress(x, self.target_ratio)
        bit_width = self.coder.num_bits if isinstance(self.coder, TrainableVDDIBCoder) else 32
        total_bits = compressed_x.numel() * bit_width
        tx_time = total_bits / self.bandwidth_bps
        self.accumulated_tx_time += tx_time

        decompressed_x = self.coder.decompress(compressed_x, self.target_ratio, original_shape)
        return decompressed_x


class DeviceNode(nn.Module):
    def __init__(self, node_id: str, weight: float, num_classes: int = 3):
        super().__init__()
        self.node_id = node_id
        self.weight = weight
        self.layers = nn.Sequential()
        self.edges = nn.ModuleDict()
        self.predecessors = []
        self.downsample_factor = 1

        # Local Auxiliary Predictor (p_psi_k) to enforce local task-relevance
        self.aux_classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(1),
            nn.Linear(1, num_classes)  # Lazily initialized
        )
        self.is_aux_initialized = False
        self.num_classes = num_classes

    def _lazy_init_aux(self, in_channels, device):
        self.aux_classifier[2] = nn.Linear(in_channels, self.num_classes).to(device)
        self.is_aux_initialized = True

    def forward_aux(self, u: torch.Tensor) -> torch.Tensor:
        in_channels = u.shape[1]
        if not self.is_aux_initialized or self.aux_classifier[2].in_features != in_channels:
            self._lazy_init_aux(in_channels, u.device)
        return self.aux_classifier(u)

    def add_successor(self, successor_node: 'DeviceNode', bandwidth_bps: float):
        edge_id = f"{self.node_id}->{successor_node.node_id}"
        self.edges[edge_id] = CommunicationEdge(self.node_id, successor_node.node_id, bandwidth_bps)
        successor_node.predecessors.append(self.node_id)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


# 3. Define Distributed Graph Coordinator
class DistributedGraph(nn.Module):
    def __init__(self, bandwidth_bps: float = 10e6):
        super().__init__()
        self.nodes = nn.ModuleDict()
        self.topological_levels = []
        self.bandwidth_bps = bandwidth_bps

    def add_node(self, node_id: str, weight: float):
        self.nodes[node_id] = DeviceNode(node_id, weight)

    def add_edge(self, source_id: str, target_id: str):
        self.nodes[source_id].add_successor(self.nodes[target_id], self.bandwidth_bps)

    def set_target_ratio(self, ratio: int):
        for node in self.nodes.values():
            for edge in node.edges.values():
                edge.target_ratio = ratio

    def set_coder_type(self, coder_class):
        for node in self.nodes.values():
            for edge in node.edges.values():
                edge.coder = coder_class()

    def get_cumulative_tx_time(self) -> float:
        total_time = 0.0
        for node in self.nodes.values():
            for edge in node.edges.values():
                total_time += edge.accumulated_tx_time
        return total_time

    def reset_tx_trackers(self):
        for node in self.nodes.values():
            for edge in node.edges.values():
                edge.reset_tracker()

    def _get_in_channels(self, module: nn.Module) -> int:
        for sub_module in module.modules():
            if isinstance(sub_module, nn.Conv2d):
                return sub_module.in_channels
        return 3

    def compile_and_allocate(self, model: nn.Module):
        in_degree = {node_id: len(node.predecessors) for node_id, node in self.nodes.items()}
        queue = deque([node_id for node_id, deg in in_degree.items() if deg == 0])

        levels = []
        while queue:
            level_nodes = list(queue)
            levels.append(level_nodes)
            queue.clear()
            for node_id in level_nodes:
                for edge in self.nodes[node_id].edges.values():
                    target_id = edge.target_id
                    in_degree[target_id] -= 1
                    if in_degree[target_id] == 0:
                        queue.append(target_id)

        self.topological_levels = levels
        D = len(levels)
        features_layers = list(model.features.children())
        num_layers = len(features_layers)
        block_size = num_layers // D

        for d, level_nodes in enumerate(levels):
            start_idx = d * block_size
            end_idx = (d + 1) * block_size if d < D - 1 else num_layers
            layer_block = nn.Sequential(*features_layers[start_idx:end_idx])

            for node_id in level_nodes:
                node = self.nodes[node_id]
                if d == D - 1:
                    node.layers = nn.Sequential(
                        layer_block,
                        model.avgpool,
                        nn.Flatten(1),
                        model.classifier
                    )
                else:
                    node.layers = layer_block

        with torch.no_grad():
            for node in self.nodes.values():
                in_channels = self._get_in_channels(node.layers)
                dummy_in = torch.randn(1, in_channels, 224, 224)
                try:
                    test_layers = node.layers[0] if isinstance(node.layers[0], nn.Sequential) else node.layers
                    dummy_out = test_layers(dummy_in)
                    node.downsample_factor = dummy_in.shape[2] // dummy_out.shape[2]
                except Exception:
                    node.downsample_factor = 1

    def forward(self, x: torch.Tensor):
        node_outputs = {}
        aux_predictions = {}

        for d, level_nodes in enumerate(self.topological_levels):
            for node_id in level_nodes:
                node = self.nodes[node_id]
                if d == 0:
                    node_input = x
                else:
                    incoming_tensors = []
                    for pred_id in node.predecessors:
                        edge_id = f"{pred_id}->{node_id}"
                        pred_out = node_outputs[pred_id]

                        transmitted_tensor = self.nodes[pred_id].edges[edge_id](pred_out)
                        incoming_tensors.append(transmitted_tensor)

                        if self.nodes[pred_id].edges[edge_id].target_ratio > 1:
                            compressed_feature = self.nodes[pred_id].edges[edge_id].coder.compress(
                                pred_out, self.nodes[pred_id].edges[edge_id].target_ratio
                            )
                            aux_predictions[pred_id] = self.nodes[pred_id].forward_aux(compressed_feature)

                    if len(incoming_tensors) == 1:
                        node_input = incoming_tensors[0]
                    else:
                        scale = node.downsample_factor
                        pad_size = self.nodes[node.predecessors[0]].downsample_factor
                        crop_rows = pad_size // scale
                        h_A = incoming_tensors[0].shape[2] - crop_rows
                        clean_A = incoming_tensors[0][:, :, 0:h_A, :]
                        clean_B = incoming_tensors[1][:, :, crop_rows:, :]
                        node_input = torch.cat([clean_A, clean_B], dim=2)

                node_outputs[node_id] = node(node_input)

                if len(node.edges) > 1:
                    successors = [self.nodes[edge.target_id] for edge in node.edges.values()]
                    total_weight = sum(succ.weight for succ in successors)
                    B, C, H, W = node_outputs[node_id].shape
                    h_A = int(round((successors[0].weight / total_weight) * H))
                    pad_size = successors[0].downsample_factor
                    slice_A = node_outputs[node_id][:, :, 0 : h_A + pad_size, :]
                    slice_B = node_outputs[node_id][:, :, h_A - pad_size : H, :]
                    node_outputs[node_id] = slice_A
                    node_outputs[f"{node_id}_split2"] = slice_B

        terminal_node_id = self.topological_levels[-1][0]
        return node_outputs[terminal_node_id], aux_predictions

In [3]:
# =====================================================================
# CELL 3: TRAINING & EVALUATION HELPER FUNCTIONS
# =====================================================================
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

def train_system(dist_graph, train_loader, epochs=2, lr=1e-4, beta=0.1, device="cpu", desc_prefix="Training"):
    dist_graph.train()

    # Force initialization of all lazy layers with a real forward pass
    dist_graph.eval()
    with torch.no_grad():
        dummy_input = torch.randn(1, 3, 224, 224).to(device)
        _, _ = dist_graph(dummy_input)
    dist_graph.train()

    # Collect unique parameters that require gradients (only the coder parameters will be collected)
    seen_params = set()
    unique_params = []

    for node in dist_graph.nodes.values():
        for p in node.layers.parameters():
            if p.requires_grad and p not in seen_params:
                seen_params.add(p)
                unique_params.append(p)
        for p in node.aux_classifier.parameters():
            if p.requires_grad and p not in seen_params:
                seen_params.add(p)
                unique_params.append(p)
        for edge in node.edges.values():
            if isinstance(edge.coder, TrainableVDDIBCoder):
                for p in edge.coder.parameters():
                    if p.requires_grad and p not in seen_params:
                        seen_params.add(p)
                        unique_params.append(p)

    optimizer = torch.optim.Adam(unique_params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        progress_bar = tqdm(train_loader, desc=f"{desc_prefix} - Epoch {epoch+1}/{epochs}", leave=True)

        for batch_idx, (images, targets) in enumerate(progress_bar):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()

            # Forward pass returns global prediction and local auxiliary predictions
            global_output, aux_outputs = dist_graph(images)

            # 1. Global Task Loss (Equation 12, Term 1)
            loss = criterion(global_output, targets)

            # 2. Local Auxiliary Task Losses (Equation 12, Term 2)
            if len(aux_outputs) > 0:
                aux_loss = 0.0
                for node_id, aux_out in aux_outputs.items():
                    aux_loss += criterion(aux_out, targets)
                loss += beta * aux_loss  # Scaled by hyperparameter beta

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = global_output.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar.set_postfix({
                'Loss': f"{running_loss/(batch_idx+1):.4f}",
                'Acc': f"{100.*correct/total:.2f}%"
            })


def evaluate_system(model_or_graph, test_loader, device):
    model_or_graph.eval()
    correct = 0
    total = 0
    if isinstance(model_or_graph, DistributedGraph):
        model_or_graph.reset_tx_trackers()

    with torch.no_grad():
        for images, targets in test_loader:
            images, targets = images.to(device), targets.to(device)
            if isinstance(model_or_graph, DistributedGraph):
                outputs, _ = model_or_graph(images)
            else:
                outputs = model_or_graph(images)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    accuracy = (correct / total) * 100
    tx_time = model_or_graph.get_cumulative_tx_time() if isinstance(model_or_graph, DistributedGraph) else 0.0
    return accuracy, tx_time

In [4]:
# =====================================================================
# CELL 4: MAIN EXPERIMENT PIPELINE
# =====================================================================
import torchvision.models as models
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running pipeline on: {device}\n")

# 1. Initialize and Modify Pretrained VGG16
print("Initializing pretrained VGG16...")
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# Freeze the feature extractor layers to prevent overfitting
for param in vgg16.features.parameters():
    param.requires_grad = False

vgg16.classifier[6] = nn.Linear(4096, 3)  # 3 target classes
vgg16 = vgg16.to(device)

# 2. Define DAG Topology (Bandwidth: 10 Mbps)
bandwidth_bps = 10e6
dist_graph = DistributedGraph(bandwidth_bps=bandwidth_bps)
dist_graph.add_node('P', weight=1.0)
dist_graph.add_node('A', weight=0.6)
dist_graph.add_node('B', weight=0.4)
dist_graph.add_node('S', weight=1.0)

dist_graph.add_edge('P', 'A')
dist_graph.add_edge('P', 'B')
dist_graph.add_edge('A', 'S')
dist_graph.add_edge('B', 'S')

dist_graph.compile_and_allocate(vgg16)
dist_graph.to(device)

# --- PHASE 1: Fine-tune Baseline Model (No Compression) ---
print("\n--- PHASE 1: Fine-tuning Baseline Model (No Compression) ---")
dist_graph.set_target_ratio(1)  # No compression
train_system(dist_graph, train_loader, epochs=2, lr=1e-4, device=device, desc_prefix="Phase 1 (Baseline)")

# Save the clean, fine-tuned baseline state to prevent weight contamination
baseline_state = copy.deepcopy(vgg16.state_dict())

# --- PHASE 2: Evaluate Baselines ---
print("\nEvaluating baselines...")
results = {}

# 1. Baseline (Unsplit Model)
acc_base, _ = evaluate_system(vgg16, test_loader, device)
results['Baseline (Unsplit)'] = (acc_base, 0.0)

# 2. Distributed (No Compression)
dist_graph.set_target_ratio(1)
acc_no_comp, tx_no_comp = evaluate_system(dist_graph, test_loader, device)
results['Distributed (No Compression)'] = (acc_no_comp, tx_no_comp)

# --- PHASE 3: Evaluate Spatial Downsampling (Aligned Compression Ratios) ---
print("\nEvaluating Spatial Downsampling...")
dist_graph.set_coder_type(SpatialDownsamplingCoder)

target_ratios = [8, 16, 32]

for ratio in target_ratios:
    dist_graph.set_target_ratio(ratio)
    acc, tx = evaluate_system(dist_graph, test_loader, device)
    results[f'Distributed (Spatial Downsampling {ratio}x CR)'] = (acc, tx)

# --- PHASE 4: Train and Evaluate Proposed VDDIB (Aligned Compression Ratios) ---
# We freeze the entire VGG16 model (both features and classifier) to evaluate the true compression capability.
# The VDDIB joint loss (Equation 12) is optimized using beta = 0.1.
for ratio in target_ratios:
    print(f"\n--- PHASE 4: Training Proposed Trainable VDDIB Coder ({ratio}x CR) ---")

    # Restore the clean baseline state
    vgg16.load_state_dict(copy.deepcopy(baseline_state))

    # Freeze the entire VGG16 model
    for param in vgg16.parameters():
        param.requires_grad = False

    # Set coder type (resets weights for the new run)
    dist_graph.set_coder_type(TrainableVDDIBCoder)
    dist_graph.set_target_ratio(ratio)

    # Train only the coder parameters and auxiliary classifiers using the joint VDDIB loss
    train_system(dist_graph, train_loader, epochs=5, lr=5e-4, beta=0.1, device=device, desc_prefix=f"Phase 4 (VDDIB {ratio}x)")

    # Evaluate immediately (preserving the trained weights)
    acc, tx = evaluate_system(dist_graph, test_loader, device)
    results[f'Distributed (Proposed Trainable VDDIB {ratio}x CR)'] = (acc, tx)

# --- PRINT FINAL COMPARISON TABLE ---
print("\n" + "="*95)
print(f"EVALUATION RESULTS ON {len(test_loader.dataset)} TEST SAMPLES (Bandwidth: {bandwidth_bps/1e6:.1f} Mbps)")
print("="*95)
print(f"{'Execution Mode':<45} | {'Accuracy (%)':<15} | {'Cumulative Tx Time (s)':<25}")
print("-"*95)
for mode, (acc, tx) in results.items():
    tx_str = f"{tx:.4f}" if tx > 0 else "N/A (Local)"
    print(f"{mode:<45} | {acc:<15.2f} | {tx_str:<25}")
print("="*95)

Running pipeline on: cuda

Initializing pretrained VGG16...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 155MB/s]



--- PHASE 1: Fine-tuning Baseline Model (No Compression) ---


Phase 1 (Baseline) - Epoch 1/2:   0%|          | 0/94 [00:00<?, ?it/s]

Phase 1 (Baseline) - Epoch 2/2:   0%|          | 0/94 [00:00<?, ?it/s]

...

Phase 4 (VDDIB 32x) - Epoch 2/5:   0%|          | 0/94 [00:00<?, ?it/s]

Phase 4 (VDDIB 32x) - Epoch 3/5:   0%|          | 0/94 [00:00<?, ?it/s]

Phase 4 (VDDIB 32x) - Epoch 4/5:   0%|          | 0/94 [00:00<?, ?it/s]

Phase 4 (VDDIB 32x) - Epoch 5/5:   0%|          | 0/94 [00:00<?, ?it/s]


EVALUATION RESULTS ON 600 TEST SAMPLES (Bandwidth: 10.0 Mbps)
Execution Mode                                | Accuracy (%)    | Cumulative Tx Time (s)   
-----------------------------------------------------------------------------------------------
Baseline (Unsplit)                            | 91.50           | N/A (Local)              
Distributed (No Compression)                  | 90.83           | 1899.2333                
Distributed (Spatial Downsampling 8x CR)      | 75.33           | 218.2349                 
Distributed (Spatial Downsampling 16x CR)     | 62.83           | 110.1005                 
Distributed (Spatial Downsampling 32x CR)     | 48.67           | 50.1350                  
Distributed (Proposed Trainable VDDIB 8x CR)  | 75.33           | 237.4042                 
Distributed (Proposed Trainable VDDIB 16x CR) | 75.33           | 118.7021                 
Distributed (Proposed Trainable VDDIB 32x CR) | 72.67           | 59.3510                  
